In [ ]:
from utils import *
from plotly.subplots import make_subplots
from tqdm.auto import tqdm

In [ ]:
def split_ratios_loader():
    results_dir = Path("../results/split_ratios")
    scalars = TBScalars(".cache/split_ratios")

    res_df = []
    for test in tqdm([*results_dir.iterdir()]):
        params = test.name.split("-")
        test_r = {}
        test_r["env"] = params[0]
        types = {"wm_ratio": int, "rl_ratio": int, "seed": int}
        for (name, typ), value in zip(types.items(), params[1:]):
            test_r[name] = typ(value.removeprefix(f"{name}="))
        df = scalars.read(test)
        df = df[df["tag"] == "val/mean_ep_ret"]
        test_r["score"] = df.iloc[-1]["value"]
        res_df.append({"path": test, **test_r})
        scalars.read(test)
    res_df = pd.DataFrame.from_records(res_df)

    return res_df, scalars


res_df, scalars = split_ratios_loader()

In [ ]:
tags = []
for _, row in res_df.iterrows():
    tags.append(f"{row['wm_ratio']}/{row['rl_ratio']}")
res_df["tag"] = tags
res_df = res_df.sort_values(by="tag")

In [ ]:
envs = res_df["env"].unique()

fig = make_subplots(
    rows=len(envs),
    row_titles=[*envs],
    cols=1,
)

for row, env in enumerate(envs, 1):
    df_ = res_df[res_df["env"] == env]
    fig.add_trace(
        go.Box(x=df_["tag"], y=df_["score"], boxmean=True, boxpoints="all"),
        row=row,
        col=1,
    )

# fig.update_traces(meanline_visible=True)
fig.update_layout(
    width=1000,
    height=400,
    xaxis_title="Configuration",
    yaxis_title="Score",
)

fig.write_image("../tex/assets/split_ratios_perf.pdf")
fig

In [ ]:
wm_ratios = res_df["wm_ratio"].unique()
rl_ratios = res_df["rl_ratio"].unique()

fig = make_subplots(
    rows=len(wm_ratios),
    row_titles=[str(x) for x in wm_ratios],
    cols=len(rl_ratios),
    column_titles=[str(x) for x in rl_ratios],
)

color = next(make_color_iter())
for row, wm_ratio in enumerate(wm_ratios, 1):
    for col, rl_ratio in enumerate(rl_ratios, 1):
        res_dfs = res_df[
            (res_df["wm_ratio"] == wm_ratio) & (res_df["rl_ratio"] == rl_ratio)
        ]
        dfs = []
        for _, test in res_dfs.iterrows():
            df = scalars.read(test["path"])
            df = df[df["tag"] == "val/mean_ep_ret"]
            df["index"] = np.arange(len(df))
            dfs.append(df)
        df = pd.concat(dfs)

        g = df.groupby("index")
        avg_df = pd.DataFrame.from_records(
            {
                "score_mean": g["value"].mean(),
                "score_std": g["value"].std(),
                "step": g["step"].median(),
            }
        )

        for trace in err_line(
            x=avg_df["step"],
            y=avg_df["score_mean"],
            std=avg_df["score_std"],
            color=color,
        ):
            fig.add_trace(trace, row=row, col=col)

fig

In [ ]:
px.violin(res_df, x="wm_ratio", y="score", log_x=True)

In [ ]:
px.violin(res_df, x="rl_ratio", y="score", log_x=True)